In [ ]:
!pip list | grep sciopy

# ISX-3 / ISX-3mini example

This notebook configures and measures an impedance spectrum with the `ISX_3` class over the USB full-speed serial interface. Connect the instrument and close the official Sciospec software before running the notebook.

In [ ]:
import cmath

import matplotlib.pyplot as plt

from sciopy import ISX_3, EisMeasurementSetup, available_serial_ports

## Connect the device

Display the available serial ports, then set `port` to the ISX-3 port. Typical names are `"COM3"` on Windows and `"/dev/ttyACM0"` or `"/dev/ttyUSB0"` on Linux.

In [ ]:
available_serial_ports()

In [ ]:
port = "/dev/ttyACM0"  # Change this to the port shown above.

isx = ISX_3()
isx.connect_device_FS(port, timeout=1)

In [ ]:
# Read startup messages and device identification.
isx.SystemMessageCallback()
isx.GetDeviceID()
isx.GetARMFirmwareID()
isx.GetFPGAFirmwareID()

## Configure the spectrum

The example uses a logarithmic sweep and a four-point measurement on the BNC input with automatic current ranging. Amplitude is specified in millivolts. Adjust these values for your sample and device configuration.

In [ ]:
setup = EisMeasurementSetup(
    start=10,  # Hz
    stop=100_000,  # Hz
    step=10,  # number of frequency points
    stepmode="log",
    avg=1,  # number of spectra
    amplitude=100,  # mV peak amplitude
    precision=1,
    measurement_time=5,  # response collection timeout in seconds
)

isx.SetMeasurementSetup(setup)

In [ ]:
# Example 1: measure through the direct BNC input.
isx.ClearFE_Settings()
isx.SetFE_Settings(
    measurement_mode="4-point",
    measurement_channel="bnc",
    current_range="auto",
    voltage_range="1v",
)

isx.GetFE_Settings()
isx.GetFrequencyCount()
isx.GetFrequencyList()

## Measure and plot

Each returned `ISXMeasurement` contains a frequency-row ID and a complex impedance in ohms. The plot uses the row IDs because those are supplied directly with each measurement frame.

In [ ]:
data = isx.StartStopMeasurement()
if not data:
    raise RuntimeError(
        "No measurements received. Check the frontend, sample, port, and timeout."
    )

for point in data:
    print(
        f"row={point.frequency_id:3d}, "
        f"Z={point.impedance.real:.6g} {point.impedance.imag:+.6g}j ohm"
    )

In [ ]:
row_ids = [point.frequency_id for point in data]
impedances = [point.impedance for point in data]

fig, (ax_magnitude, ax_phase) = plt.subplots(2, 1, sharex=True)
ax_magnitude.semilogy(row_ids, [abs(z) for z in impedances], "o-")
ax_magnitude.set_ylabel("|Z| in $\Omega$")
ax_magnitude.grid(True)

ax_phase.plot(row_ids, [cmath.phase(z) for z in impedances], "o-")
ax_phase.set(xlabel="Frequency-row ID", ylabel="Phase in rad")
ax_phase.grid(True)
fig.suptitle("ISX-3 impedance spectrum")
fig.tight_layout()
plt.show()

## Example 2: direct extension-port measurement

Use `measurement_channel="extension"` for a direct measurement through the extension interface without mux routing. The frequency setup configured above is reused.

In [ ]:
isx.ClearFE_Settings()
isx.SetFE_Settings(
    measurement_mode="4-point",
    measurement_channel="extension",
    current_range="auto",
    voltage_range="1v",
)
isx.GetFE_Settings()

In [ ]:
extension_data = isx.StartStopMeasurement()
if not extension_data:
    raise RuntimeError("No direct extension-port measurements received.")

for point in extension_data:
    print(
        f"row={point.frequency_id:3d}, "
        f"Z={point.impedance.real:.6g} {point.impedance.imag:+.6g}j ohm"
    )

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(
    [point.frequency_id for point in extension_data],
    [abs(point.impedance) for point in extension_data],
    "o-",
)
ax.set(
    xlabel="Frequency-row ID",
    ylabel="|Z| [ohm]",
    title="ISX-3 direct extension-port spectrum",
)
ax.grid(True)
plt.show()

## Example 3: multiplexer measurement

Use `measurement_channel="mux"` when a compatible Sciospec multiplexer is connected. First query the installed module, then define the desired `(C, R, WS, W)` channel routings. Channel availability and numbering depend on the installed module, so adapt the example routings to your hardware.

### Extension and internal module identifiers

`GetExtensionPortModule()` returns both an extension-module identifier and an internal-module identifier. These tables follow the ISX-3 / ISX-3mini manual, revision 108 (2024-09-27), section 6.5.9, **Get ExtensionPort Module**. Where the manual provides only an identifier name and no functional explanation, that limitation is stated explicitly.

#### Extension modules

| Code | Module returned by the device | Description from the manual |
| --- | --- | --- |
| `0x00` | No module connected | No external extension module is detected. |
| `0x01` | `MEArack` | Sciospec MEArack identifier. Section 7.5 states that the MEArack is compatible with MUX64 and CSX-64; its first pin is marked `1`. |
| `0x02` | `MuxModule32` | External module identifier named MuxModule32. Section 6.5.9 gives no additional functional description for this specific identifier. |
| `0x03` | `ECIS Adapter` | External ECIS Adapter identifier. Section 6.5.9 gives no additional functional description. |
| `0x05` | `ExtensionPortAdapter` | External ExtensionPortAdapter identifier. Section 6.5.9 gives no additional functional description. |
| `0x06` | `SlideChipAdapter` | External SlideChipAdapter identifier. Section 6.5.9 gives no additional functional description. |
| `0x07` | `Mux32any2any` (external) | External any-to-any mux identifier. The MUX32/MUX64 section describes a 32- or 64-channel mux in which every channel can be assigned as Counter, Reference, Working Sense, or Work for 2-, 3-, or 4-point measurements; measurements are acquired sequentially. |
| `0x08` | `DaQEisMux` | External DaQEisMux identifier. Section 6.5.9 gives no additional functional description. |
| `0x09` | `Mux32any2any2202` (external) | External any-to-any mux identifier with a device-reported channel count. The response includes an additional two-byte unsigned external channel count for this code. |

There is no extension-module code `0x04` in the revision 108 table.

#### Internal modules

| Code | Module returned by the device | Description from the manual |
| --- | --- | --- |
| `0x00` | No module connected | No internal module is detected. |
| `0x01` | `MuxModule16x4` | Internal module identifier named MuxModule16x4. Section 6.5.9 gives no additional functional description for this specific identifier. |
| `0x02` | `MuxModule32x2` | Internal module identifier named MuxModule32x2. Section 6.5.9 gives no additional functional description for this specific identifier. |
| `0x07` | `Mux32any2any` (internal) | Internal any-to-any mux identifier. For the general MUX32/MUX64 behavior, see the Counter/Reference/Working Sense/Work description above. |
| `0x09` | `Mux32any2any2202` (internal) | Internal any-to-any mux identifier with a device-reported channel count. The response includes an additional two-byte unsigned internal channel count for this code. |

The response length is two bytes when neither module is `0x09`, four bytes when either module is `0x09`, and six bytes when both are `0x09`. Each optional channel count occupies two bytes.

In [ ]:
# Identify the connected external and internal mux modules.
mux_module = isx.GetExtensionPortModule()
print(mux_module)

In [ ]:
# Replace these example channel assignments with valid routings for your mux.
mux_routings = {
    "routing 1": (1, 2, 3, 4),  # C, R, WS, W
    "routing 2": (5, 6, 7, 8),
}

# Select the dedicated mux measurement channel.
isx.ClearFE_Settings()
isx.SetFE_Settings(
    measurement_mode="4-point",
    measurement_channel="mux",
    current_range="auto",
    voltage_range="1v",
)

In [ ]:
mux_data = {}

for name, (counter, reference, working_sense, working) in mux_routings.items():
    isx.SetExtensionPortChannel(counter, reference, working_sense, working)
    isx.GetExtensionPortChannel()  # Read back the active C/R/WS/W routing.
    points = isx.StartStopMeasurement()
    if not points:
        raise RuntimeError(f"No measurements received for {name}.")
    mux_data[name] = points
    print(f"{name}: received {len(points)} points")

In [ ]:
fig, ax = plt.subplots()
for name, points in mux_data.items():
    ax.semilogy(
        [point.frequency_id for point in points],
        [abs(point.impedance) for point in points],
        "o-",
        label=name,
    )

ax.set(
    xlabel="Frequency-row ID",
    ylabel="|Z| in $\Omega$",
    title="ISX-3 mux measurements",
)
ax.grid(True)
ax.legend()
plt.show()

## Disconnect

Close the serial connection before reconnecting with this notebook or the official software.

In [ ]:
isx.disconnect_device()